# Implement SmolLM-135M from Scratch

### Problem Statement
[SmolLM-135M](https://huggingface.co/HuggingFaceTB/SmolLM-135M) is a small decoder-only language model with the Llama architecture. Build it from scratch in PyTorch, so that the pretrained weights load into your modules and your logits match the reference `transformers` model.

### Architecture (from the checkpoint's `config.json`)

| | |
|---|---|
| vocabulary | 49,152 tokens |
| hidden size | 576 |
| decoder layers | 30 |
| attention | 9 query heads, 3 key/value heads (grouped-query attention), head_dim 64, no biases |
| positions | RoPE, base θ = 10000, `rotate_half` (half-split) pairing |
| MLP | SwiGLU: `W_down(silu(W_gate(x)) * W_up(x))`, intermediate size 1536 |
| normalization | RMSNorm, eps 1e-5, **pre-norm** (before attention and before the MLP), plus a final norm |
| output | `lm_head` tied to the input embedding (`tie_word_embeddings`) |

That is about 134.5M parameters (the tied embedding counted once). SmolLM**2**-135M has the same shapes but RoPE base 100000.

### Requirements
1. **Attention block** (graded with `check("smollm", ...)`): `rotate_half`, `apply_rotary_pos_emb`, `repeat_kv` (KV head `i // n_rep` serves query head `i`), `RotaryEmbedder` and `RopeAttention`. `RopeAttention.forward(hidden_states, attention_mask)` takes an **additive** float mask broadcastable to `(batch, heads, seq, seq)`: 0 where attending is allowed, a very negative number where it is not.
2. **The rest of the model**: `MLP`, `RMSNorm`, `LlamaDecoder` (residual pre-norm block), `smolModel` (embeddings → layers → final norm; it builds the causal + padding mask once from the `(batch, seq)` attention mask) and `smolLM` (adds the tied `lm_head` and returns `{"logits": ...}`).
3. Use the module names below, so that `load_hf_weights` can copy the pretrained weights in.
4. **Test**: a quick CPU test with a tiny config, then load the real weights and check that your logits match `transformers`' to ~1e-4 and that greedy decoding produces the same text.

```
smolLM(
  (model): smolModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoder(
        (self_attn): RopeAttention(
          (W_query): Linear(in_features=576, out_features=576, bias=False)
          (W_key): Linear(in_features=576, out_features=192, bias=False)
          (W_value): Linear(in_features=576, out_features=192, bias=False)
          (W_output): Linear(in_features=576, out_features=576, bias=False)
          (rotary_emb): RotaryEmbedder()
        )
        (mlp): MLP(
          (W_gate): Linear(in_features=576, out_features=1536, bias=False)
          (W_up): Linear(in_features=576, out_features=1536, bias=False)
          (W_down): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLU()
        )
        (pre_attn_rmsnorm): RMSNorm()
        (pre_mlp_rmsnorm): RMSNorm()
      )
    )
    (norm): RMSNorm()
  )
  (lm_head): Linear(in_features=576, out_features=49152, bias=False)
)
```

## Solution

In [ ]:
import math
import torch
from torch import nn
import torch.nn.functional as F

# Helper function to rotate the last half of a tensor
# Used in rotary positional embeddings to compute sine and cosine rotations
def rotate_half(x):
    # Split tensor into two halves along the last dimension
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    # Rotate: negate the second half and concatenate it with the first half
    return torch.cat((-x2, x1), dim=-1)

# Applies rotary positional embeddings to query (q) and key (k) tensors
# q: (batch, num_heads, seq, head_dim), k: (batch, kv_heads, seq, head_dim)
def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    # cos/sin: (batch, seq, head_dim) -> (batch, 1, seq, head_dim) to broadcast over heads
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)

    # Apply rotations to q and k using cos and sin
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

# Repeats key-value tensors for multiple attention heads
# (batch, kv_heads, seq, head_dim) -> (batch, kv_heads * n_rep, seq, head_dim), where
# output head i is KV head i // n_rep (same as repeat_interleave on dim 1)
def repeat_kv(hidden_states, n_rep):
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    # Expand the number of key-value heads by repeating them (a view, no copy yet)
    hidden_states = hidden_states[:, :, None, :, :].expand(
        batch, num_key_value_heads, n_rep, slen, head_dim
    )
    # Reshape to align with the expected multi-head attention format
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# Computes rotary positional embeddings for queries and keys
class RotaryEmbedder(nn.Module):
    def __init__(self, dim, base):
        super().__init__()
        # Inverse frequencies theta_i = base^(-2i/dim); a buffer so .to(device) moves it
        freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("freq", freq, persistent=False)

    @torch.no_grad()
    def forward(self, x):
        # Positions 0 .. seq_len-1 of x: (..., seq_len, head_dim)
        pos = torch.arange(x.shape[-2], device=x.device, dtype=torch.float32)
        # Angles m * theta_i: (1, seq_len, head_dim / 2)
        angles = torch.outer(pos, self.freq).unsqueeze(0)
        # Pair i shares its angle with i + head_dim/2, matching rotate_half
        emb = torch.cat((angles, angles), dim=-1)
        return emb.cos().to(x.dtype), emb.sin().to(x.dtype)

# Implements attention with rotary positional embeddings and grouped-query K/V heads
class RopeAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Model dimensions and attention configurations
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_heads
        self.head_dim = config.hidden_size // self.num_heads
        self.kv_heads = config.kv_heads  # Number of key-value heads
        self.rope_theta = getattr(config, "rope_theta", 10000.0)  # RoPE base

        # Linear projections for queries, keys, values, and output
        self.W_query = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.W_key = nn.Linear(config.hidden_size, self.kv_heads * self.head_dim, bias=False)
        self.W_value = nn.Linear(config.hidden_size, self.kv_heads * self.head_dim, bias=False)
        self.W_output = nn.Linear(self.num_heads * self.head_dim, config.hidden_size, bias=False)

        # Rotary embedding generator
        self.rotary_emb = RotaryEmbedder(base=self.rope_theta, dim=self.head_dim)

    def forward(self, hidden_states: torch.Tensor, attention_mask=None):
        """attention_mask: additive float mask broadcastable to (batch, heads, seq, seq),
        0 where attending is allowed and a large negative number (or -inf) elsewhere."""
        # Input dimensions: (batch_size, seq_len, hidden_size)
        b, q, _ = hidden_states.size()

        # Project input hidden states into queries, keys, and values
        q_states = self.W_query(hidden_states)
        k_states = self.W_key(hidden_states)
        v_states = self.W_value(hidden_states)

        # Reshape and transpose for multi-head attention: (b, heads, q, head_dim)
        q_states = q_states.view(b, q, self.num_heads, self.head_dim).transpose(1, 2)
        k_states = k_states.view(b, q, self.kv_heads, self.head_dim).transpose(1, 2)
        v_states = v_states.view(b, q, self.kv_heads, self.head_dim).transpose(1, 2)

        # Rotate queries and keys by their positions (never the values)
        cos, sin = self.rotary_emb(q_states)
        q_states, k_states = apply_rotary_pos_emb(q_states, k_states, cos, sin)

        # Share each K/V head with its group of query heads
        kv_groups = self.num_heads // self.kv_heads
        k_states = repeat_kv(k_states, kv_groups)
        v_states = repeat_kv(v_states, kv_groups)

        # Scaled dot-product attention with the additive (causal / padding) mask
        attn_weights = torch.matmul(q_states, k_states.transpose(2, 3)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, v_states)

        # Merge the heads back: (b, q, num_heads * head_dim), then project
        attn_output = attn_output.transpose(1, 2).reshape(b, q, -1)
        return self.W_output(attn_output)

In [ ]:
# Gated MLP (SwiGLU): down(silu(gate(x)) * up(x))
class MLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.W_gate = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.W_up = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.W_down = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        return self.W_down(self.act_fn(self.W_gate(x)) * self.W_up(x))


# RMSNorm: x / sqrt(mean(x^2) + eps) * weight
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))  # Learnable scaling factor
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.float()  # reduce in fp32, as Llama does
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)


# One pre-norm decoder block: x + attn(norm(x)), then x + mlp(norm(x))
class LlamaDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = RopeAttention(config)
        self.mlp = MLP(hidden_size=config.hidden_size, intermediate_size=config.intermediate_size)
        self.pre_attn_rmsnorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.pre_mlp_rmsnorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, attention_mask=None):
        # attention_mask: the additive (batch, 1, seq, seq) mask built once by smolModel
        hidden_states = hidden_states + self.self_attn(
            self.pre_attn_rmsnorm(hidden_states), attention_mask=attention_mask)
        hidden_states = hidden_states + self.mlp(self.pre_mlp_rmsnorm(hidden_states))
        return hidden_states

In [ ]:
# The main model: token embeddings -> decoder stack -> final RMSNorm
class smolModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([LlamaDecoder(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, input_ids, attention_mask=None):
        hidden_states = self.embed_tokens(input_ids)
        b, s, _ = hidden_states.shape

        # Additive mask, built once for all layers: position i may attend to key j
        # if j <= i (causal) and key j is not padding. A large negative number rather
        # than -inf keeps fully masked (padding) rows from turning into NaN.
        allowed = torch.ones(s, s, dtype=torch.bool, device=input_ids.device).tril()
        allowed = allowed[None, None]                                      # (1, 1, s, s)
        if attention_mask is not None:                                      # (b, s), 1 = real token
            allowed = allowed & attention_mask[:, None, None, :].bool()    # (b, 1, s, s)
        mask = torch.zeros(allowed.shape, dtype=hidden_states.dtype, device=input_ids.device)
        mask = mask.masked_fill(~allowed, torch.finfo(hidden_states.dtype).min)

        for decoder_layer in self.layers:
            hidden_states = decoder_layer(hidden_states, attention_mask=mask)
        return self.norm(hidden_states)


# The complete language model: smolModel + LM head tied to the token embeddings
class smolLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.model = smolModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        # SmolLM ties the input embedding and the output projection (tie_word_embeddings)
        self.lm_head.weight = self.model.embed_tokens.weight

    def forward(self, input_ids, attention_mask=None):
        hidden_states = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.lm_head(hidden_states).float()
        return {"logits": logits}


class smolConfig:
    # SmolLM-135M (HuggingFaceTB/SmolLM-135M config.json)
    vocab_size = 49152
    hidden_size = 576
    intermediate_size = 1536
    num_hidden_layers = 30
    num_heads = 9          # num_attention_heads -> head_dim = 64
    kv_heads = 3           # num_key_value_heads (GQA, 3 query heads per K/V head)
    rope_theta = 10000.0
    rms_norm_eps = 1e-5

## Test

In [ ]:
# Quick CPU checks with a tiny config (no download needed)
class TinyConfig(smolConfig):
    vocab_size, hidden_size, intermediate_size = 100, 48, 96
    num_hidden_layers, num_heads, kv_heads = 2, 6, 2

torch.manual_seed(0)
tiny = smolLM(TinyConfig()).eval()
ids = torch.randint(0, 100, (2, 10))
with torch.no_grad():
    logits = tiny(ids)["logits"]
    assert logits.shape == (2, 10, 100)

    # Causal: changing the last token must not change any earlier logits.
    ids2 = ids.clone()
    ids2[:, -1] = (ids2[:, -1] + 1) % 100
    assert torch.allclose(tiny(ids2)["logits"][:, :-1], logits[:, :-1], atol=1e-5)

    # Padding: left-pad row 0 with 3 masked tokens. Its real tokens now sit at
    # positions 3..12, but RoPE only sees relative offsets and the padding is
    # masked out, so their logits must not change.
    padded = torch.cat([torch.zeros(1, 3, dtype=torch.long), ids[:1]], dim=1)
    mask = torch.cat([torch.zeros(1, 3), torch.ones(1, 10)], dim=1).long()
    assert torch.allclose(tiny(padded, mask)["logits"][:, 3:], logits[:1], atol=1e-4)
assert tiny.lm_head.weight is tiny.model.embed_tokens.weight, "lm_head must be tied"

n_params = sum(p.numel() for p in smolLM(smolConfig()).parameters())
print(f"SmolLM-135M has {n_params / 1e6:.1f}M parameters")  # 134.5M
print("tiny model: shapes, causal mask, padding mask and tied weights OK")

In [ ]:
# Load the pretrained SmolLM-135M weights into our model and compare with transformers.
# Needs `pip install transformers` and a ~270 MB download.
from transformers import AutoTokenizer, AutoModelForCausalLM

# Hugging Face Llama parameter names -> the names used in this notebook
RENAME = {
    "self_attn.q_proj": "self_attn.W_query", "self_attn.k_proj": "self_attn.W_key",
    "self_attn.v_proj": "self_attn.W_value", "self_attn.o_proj": "self_attn.W_output",
    "mlp.gate_proj": "mlp.W_gate", "mlp.up_proj": "mlp.W_up", "mlp.down_proj": "mlp.W_down",
    "input_layernorm": "pre_attn_rmsnorm", "post_attention_layernorm": "pre_mlp_rmsnorm",
}

def load_hf_weights(model, hf_model):
    state = {}
    for name, tensor in hf_model.state_dict().items():
        for old, new in RENAME.items():
            name = name.replace(old, new)
        state[name] = tensor
    state.setdefault("lm_head.weight", state["model.embed_tokens.weight"])  # tied
    model.load_state_dict(state, strict=True)  # every weight must find its slot

checkpoint = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
reference_model = AutoModelForCausalLM.from_pretrained(checkpoint).float().eval()

test_model = smolLM(smolConfig()).eval()
load_hf_weights(test_model, reference_model)

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    ours = test_model(**inputs)["logits"]
    ref = reference_model(**inputs).logits
print(f"max |logit difference|: {(ours - ref).abs().max().item():.2e}")
assert torch.allclose(ours, ref, atol=1e-3), "logits do not match the reference model"


@torch.no_grad()
def greedy_generate(model, input_ids, max_new_tokens=10):
    for _ in range(max_new_tokens):
        next_id = model(input_ids)["logits"][:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=-1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return input_ids

ours_text = tokenizer.decode(greedy_generate(test_model, inputs["input_ids"])[0])
ref_text = tokenizer.decode(greedy_generate(lambda ids: {"logits": reference_model(ids).logits},
                                            inputs["input_ids"])[0])
print("ours:     ", repr(ours_text))
print("reference:", repr(ref_text))
assert ours_text == ref_text